# Two-type composition sweep

Empirical case study for the two-type model. Service values and service-time
moments come from the SGD experiment (`sgd_calibration.load_calibration`).
For each patient-to-impatient ratio $r=\lambda_2/\lambda_1$, holding the total
arrival rate $\lambda$ fixed, we evaluate all 86 policies (a depth in
$\mathcal{X}$ or $x=0$ for each type; FCFS, or nonpreemptive priority to either
type) with exact stationary M/G/1 formulas. The centralized
optimum is the welfare maximizer; the optimal incentive-compatible menu is the
welfare maximizer among policies with $w_1 \le w_2$ (Lemma 3). Depth $x=0$ is the
paper's no-service option ($w=0$); in the figures a type at $x=0$ is shown as
prioritized, since it has the shortest sojourn time. The three scheduling rules
use type information only and are a restriction of the policies allowed in the
paper's centralized problem. No API calls and no
simulation.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sgd_calibration import load_calibration

cal = load_calibration(proxy="cost")
VALUES, MEANS, SECONDS = cal["values"], cal["means"], cal["seconds"]
EFFORTS = list(cal["efforts"])
ZERO = -1                  # x = 0: zero value, zero service time, top priority
print("values ", np.round(VALUES, 3))
print("means  ", np.round(MEANS, 4))
print("E[S^2] ", np.round(SECONDS, 4))

## Parameters

Type 1 is impatient, type 2 patient. The total arrival rate is fixed at
`LAMBDA`; the sweep variable is $r=\lambda_2/\lambda_1$, with
$\lambda_1=\lambda/(1+r)$ and $\lambda_2=\lambda r/(1+r)$. Delay costs are per
normalized time unit.

In [ ]:
LAMBDA = 1.0
THETA1, THETA2 = 0.0175, 0.017
RATIOS = np.logspace(-1.5, 2.0, 1401)          # lambda_2 / lambda_1
OUTPUT = Path("results") / "two_type_composition"
RULES_BOTH = ("1NP", "2NP", "FCFS")   # both types receive positive depth

## Exact stationary sojourn times

For types with positive depth, arrival rates $\lambda_i$, means $m_i$ and second
moments $b_i$, let $\rho_i=\lambda_i m_i$ and $R=\tfrac12\sum_i\lambda_i b_i$.
FCFS: $w_i = m_i + R/(1-\rho)$. Nonpreemptive priority to $h$ over $\ell$
(Cobham): $w_h = m_h + R/(1-\rho_h)$, $w_\ell = m_\ell + R/((1-\rho_h)(1-\rho))$.
A single type with positive depth is an M/G/1 FCFS queue. A type with $x=0$ has $w_i=0$.

In [ ]:
def sojourn(d1, d2, lam1, lam2, rule):
    """Return (w1, w2), or None if the assignment is unstable."""
    adm = (d1 >= 0, d2 >= 0)
    lam = (lam1 * adm[0], lam2 * adm[1])
    m = (MEANS[d1] if adm[0] else 0.0, MEANS[d2] if adm[1] else 0.0)
    b = (SECONDS[d1] if adm[0] else 0.0, SECONDS[d2] if adm[1] else 0.0)
    rho = (lam[0] * m[0], lam[1] * m[1])
    total = rho[0] + rho[1]
    if total >= 1.0:
        return None
    R = 0.5 * (lam[0] * b[0] + lam[1] * b[1])
    w = [0.0, 0.0]
    if rule == "FCFS" or not (adm[0] and adm[1]):
        for i in range(2):
            if adm[i]:
                w[i] = m[i] + R / (1.0 - total)
        return tuple(w)
    h, l = (0, 1) if rule[0] == "1" else (1, 0)      # nonpreemptive priority to type h
    w[h] = m[h] + R / (1.0 - rho[h])
    w[l] = m[l] + R / ((1.0 - rho[h]) * (1.0 - total))
    return tuple(w)


POLICIES = [(d1, d2, rule)
            for d1, d2 in itertools.product(range(-1, 5), repeat=2)
            for rule in (RULES_BOTH if (d1 >= 0 and d2 >= 0) else ("FCFS",))]
assert len(POLICIES) == 86


def optima(ratio):
    """Centralized optimum and optimal IC menu (w1 <= w2) at composition ratio."""
    lam1, lam2 = LAMBDA / (1.0 + ratio), LAMBDA * ratio / (1.0 + ratio)
    central = ic = None
    for d1, d2, rule in POLICIES:
        w = sojourn(d1, d2, lam1, lam2, rule)
        if w is None:
            continue
        W = 0.0
        if d1 >= 0:
            W += lam1 * (VALUES[d1] - THETA1 * w[0])
        if d2 >= 0:
            W += lam2 * (VALUES[d2] - THETA2 * w[1])
        rec = dict(d1=d1, d2=d2, rule=rule, welfare=W, w1=w[0], w2=w[1])
        if central is None or W > central["welfare"]:
            central = rec
        if w[0] <= w[1] + 1e-12 and (ic is None or W > ic["welfare"]):
            ic = rec
    return central, ic

## Sweep over the patient-to-impatient ratio

In [ ]:
central, ic = zip(*(optima(r) for r in RATIOS))
W_c = np.array([c["welfare"] for c in central])
W_ic = np.array([m["welfare"] for m in ic])
D1 = np.array([c["d1"] for c in central])
D2 = np.array([c["d2"] for c in central])
RULE = np.array([c["rule"] for c in central])
gap = np.where(W_c > 0, (W_c - W_ic) / np.maximum(W_c, 1e-300), 0.0)


def depth_label(d):
    return "x = 0" if d < 0 else EFFORTS[d]


def priority_label(d1, d2, rule):
    """Who is served first. Zero depth is instantaneous service, hence top priority."""
    if d1 < 0 and d2 < 0:
        return "closed"
    if d2 < 0 or rule == "2NP":
        return "patient first"
    if d1 < 0 or rule == "1NP":
        return "impatient first"
    return "FCFS"


PRIORITY = np.array([priority_label(d1, d2, r) for d1, d2, r in zip(D1, D2, RULE)])
inversion = PRIORITY == "patient first"


def intervals(labels):
    """[(left, right, label)] on which `labels` is constant."""
    out, start = [], 0
    for i in range(1, len(labels) + 1):
        if i == len(labels) or labels[i] != labels[start]:
            out.append((RATIOS[start], RATIOS[i - 1], labels[start]))
            start = i
    return out


for left, right, (d1, d2, rule) in intervals(list(zip(D1, D2, RULE))):
    sel = (RATIOS >= left) & (RATIOS <= right)
    print(f"r in [{left:7.3f}, {right:7.3f}]  impatient={depth_label(d1):6s} "
          f"patient={depth_label(d2):6s} rule={rule:4s} priority={priority_label(d1, d2, rule):15s} "
          f"max welfare gap={100 * gap[sel].max():5.1f}%")

## Figure 1: welfare

In [ ]:
RED = "#c0392b"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "axes.linewidth": 0.6, "lines.linewidth": 1.5})
lower, upper = RATIOS[0], RATIOS[-1]
inv_spans = [(l, r) for l, r, lab in intervals(list(inversion)) if lab]

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(RATIOS, W_c, color="#222222", linewidth=2.0, label="Centralized optimum")
ax.plot(RATIOS, W_ic, color=RED, linestyle="--", linewidth=2.0, label="Optimal IC menu")
ax.set_xscale("log")
ax.set_xlim(lower, upper)
lo, hi = min(W_c.min(), W_ic.min()), max(W_c.max(), W_ic.max())
ax.set_ylim(lo - 0.05 * (hi - lo), hi + 0.16 * (hi - lo))   # headroom so the legend does not cover the curves
ax.set_xlabel(r"Patient-to-impatient ratio $\lambda_2/\lambda_1$", fontsize=24)
ax.set_ylabel(r"$\mathcal{W}$", fontsize=26, rotation=0, labelpad=18)
ax.yaxis.set_label_coords(-0.2, 0.5)
ax.tick_params(labelsize=18)
ax.legend(fontsize=15, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.6, borderpad=0.4, columnspacing=1.2)
ax.grid(axis="y", alpha=0.18)

# Inset: magnify the inversion window, where the gap is too small to see.
zoom = (RATIOS >= 1.25) & (RATIOS <= 2.65)
axins = ax.inset_axes([0.315, 0.72, 0.255, 0.22])
axins.plot(RATIOS[zoom], W_c[zoom], color="#222222", linewidth=1.8)
axins.plot(RATIOS[zoom], W_ic[zoom], color=RED, linestyle="--", linewidth=1.8)
axins.set_xscale("log")
axins.set_xlim(1.25, 2.65)
axins.set_ylim(0.2965, 0.3005)
axins.set_xticks([1.5, 2.0, 2.5], ["1.5", "2", "2.5"])
axins.set_yticks([0.297, 0.299])
axins.minorticks_off()
axins.tick_params(labelsize=12, length=2, pad=1)
axins.grid(axis="y", alpha=0.18)
for side in ("top", "right"):
    axins.spines[side].set_visible(True)
ax.indicate_inset_zoom(axins, edgecolor="#555555", linewidth=1.2, alpha=1.0)

fig.subplots_adjust(left=0.21, right=0.96, top=0.87, bottom=0.21)
fig.savefig(OUTPUT.with_name("two_type_composition_welfare.png"), dpi=300)
fig.savefig(OUTPUT.with_name("two_type_composition_welfare.pdf"))
plt.show()

## Figure 2: optimal service depth of each type

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for k, (l, r) in enumerate(inv_spans):
    ax.axvspan(l, r, color="#c7e9c0", alpha=0.6, linewidth=0,
               label="Patient first" if k == 0 else None)
ax.step(RATIOS, D1 + 1, where="post", color="#222222", linewidth=2.0, label="Impatient $x_1$")
ax.step(RATIOS, D2 + 1, where="post", color=RED, linestyle="--", linewidth=2.0, label="Patient $x_2$")
ax.set_xscale("log")
ax.set_xlim(lower, upper)
ax.set_ylim(-0.4, 5.5)
ax.set_yticks(range(6), [f"$d_{k}$" for k in range(6)])
ax.set_xlabel(r"Patient-to-impatient ratio $\lambda_2/\lambda_1$", fontsize=24)
ax.tick_params(labelsize=18)
ax.legend(fontsize=15, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.4, borderpad=0.4, columnspacing=1.0)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.13, right=0.96, top=0.87, bottom=0.21)
fig.savefig(OUTPUT.with_name("two_type_composition_policy.png"), dpi=300)
fig.savefig(OUTPUT.with_name("two_type_composition_policy.pdf"))
plt.show()

## Figure 3: optimal service depth of each type under the optimal IC menu

In [ ]:
D1_ic = np.array([m["d1"] for m in ic])
D2_ic = np.array([m["d2"] for m in ic])

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.step(RATIOS, D1_ic + 1, where="post", color="#222222", linewidth=2.0, label="Impatient $x_1$")
ax.step(RATIOS, D2_ic + 1, where="post", color=RED, linestyle="--", linewidth=2.0, label="Patient $x_2$")
ax.set_xscale("log")
ax.set_xlim(lower, upper)
ax.set_ylim(-0.4, 5.5)
ax.set_yticks(range(6), [f"$d_{k}$" for k in range(6)])
ax.set_xlabel(r"Patient-to-impatient ratio $\lambda_2/\lambda_1$", fontsize=24)
ax.tick_params(labelsize=18)
ax.legend(fontsize=16, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=True,
          edgecolor="#cccccc", framealpha=1.0, handlelength=1.4, borderpad=0.4, columnspacing=1.5)
ax.grid(axis="y", alpha=0.18)
fig.subplots_adjust(left=0.13, right=0.96, top=0.87, bottom=0.21)
fig.savefig(OUTPUT.with_name("two_type_composition_policy_ic.png"), dpi=300)
fig.savefig(OUTPUT.with_name("two_type_composition_policy_ic.pdf"))
plt.show()

## Self-checks

Single-type limits reproduce the M/G/1 FCFS formula; work-conserving
nonpreemptive rules satisfy the conservation identity
$\sum_i \rho_i (w_i - m_i) = \text{const}$; overloaded assignments are rejected;
the IC optimum never exceeds the centralized optimum.

In [ ]:
lam1, lam2 = 0.3, 0.5
for d in range(5):
    w = sojourn(d, ZERO, lam1, lam2, "FCFS")
    pk = MEANS[d] + lam1 * SECONDS[d] / (2 * (1 - lam1 * MEANS[d]))
    assert abs(w[0] - pk) < 1e-12 and w[1] == 0.0
d1, d2 = 2, 3
rho = (lam1 * MEANS[d1], lam2 * MEANS[d2])
conserved = []
for r in ("FCFS", "1NP", "2NP"):
    w = sojourn(d1, d2, lam1, lam2, r)
    conserved.append(rho[0] * (w[0] - MEANS[d1]) + rho[1] * (w[1] - MEANS[d2]))
assert np.allclose(conserved, conserved[0])
assert sojourn(4, 4, 0.6, 0.6, "FCFS") is None
assert np.all(W_ic <= W_c + 1e-12)
print("all checks passed")